In [2]:
# Cell 1 — Imports + configuration

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project root = parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent

# Add src/ to Python path
sys.path.append(str(PROJECT_ROOT / "src"))

from swat_gnn.data.loader import (
    load_config,
    load_training_data,
    load_validation_data,
    load_test_data,
)

# Give the loader the absolute path to the config file
CONFIG_PATH = PROJECT_ROOT / "configs" / "data.yaml"

config = load_config(str(CONFIG_PATH))

print("Project root:", PROJECT_ROOT)
print("Dataset:", config["dataset"]["name"])
print("Root:", config["dataset"]["root"])

Project root: c:\Users\adima\Projects\industrial-dynamic-graph-anomaly-detection
Dataset: HAI_23.05
Root: data/raw/HAI_Kaggle/hai-23.05


In [4]:
# Cell 2 — Load the dataset

training_segments = load_training_data(config)
validation_segments = load_validation_data(config)
test_sensors, test_labels = load_test_data(config)

print("Training segments:", len(training_segments))
print("Validation segments:", len(validation_segments))
print("Test segments:", len(test_sensors))

for i, df in enumerate(training_segments, 1):
    print(f"Train {i}: {df.shape}")

print(f"Validation: {validation_segments[0].shape}")

for i, df in enumerate(test_sensors, 1):
    print(f"Test {i}: {df.shape}, labels: {test_labels[i-1].shape}")

Training segments: 3
Validation segments: 1
Test segments: 2
Train 1: (280800, 87)
Train 2: (291600, 87)
Train 3: (126000, 87)
Validation: (198000, 87)
Test 1: (54000, 87), labels: (54000,)
Test 2: (230400, 87), labels: (230400,)


In [ ]:
# Check timestamp integrity

datasets = [
    ("train1", training_segments[0]),
    ("train2", training_segments[1]),
    ("train3", training_segments[2]),
    ("validation", validation_segments[0]),
    ("test1", test_sensors[0]),
    ("test2", test_sensors[1]),
]

for name, df in datasets:
    timestamps = df["timestamp"]

    time_diff = timestamps.diff().dropna()

    print(f"\n{name}")
    print("-" * 50)
    print("Start:", timestamps.iloc[0])
    print("End:  ", timestamps.iloc[-1])
    print("Rows:", len(df))
    print("Duplicate timestamps:", timestamps.duplicated().sum())
    print("Missing timestamps:", timestamps.isna().sum())
    print("Most common intervals:")
    print(time_diff.value_counts().head(3))

In [ ]:
# Check missing and infinite sensor values

for name, df in datasets:
    sensor_data = df.drop(columns="timestamp")

    missing = sensor_data.isna().sum().sum()
    infinite = np.isinf(sensor_data.to_numpy()).sum()

    print(
        f"{name}: "
        f"missing={missing}, "
        f"infinite={infinite}"
    )

In [ ]:
# Verify that all datasets contain the same sensor variables

reference_features = [
    c for c in training_segments[0].columns
    if c != "timestamp"
]

print("Number of sensor variables:", len(reference_features))

for name, df in datasets:
    features = [c for c in df.columns if c != "timestamp"]

    print(
        f"{name}:",
        "MATCH" if features == reference_features else "MISMATCH"
    )

In [ ]:
# Basic statistical summary of the training data

train1 = training_segments[0]

sensor_columns = [
    c for c in train1.columns
    if c != "timestamp"
]

summary = train1[sensor_columns].describe().T

summary.head(20)

In [ ]:
# Identify constant variables across all training data

all_training_values = pd.concat(
    [df[sensor_columns] for df in training_segments],
    axis=0,
    ignore_index=True,
)

training_std = all_training_values.std()

constant_features = training_std[training_std == 0].index.tolist()

print("Total features:", len(sensor_columns))
print("Constant features:", len(constant_features))

print("\nConstant features:")
for feature in constant_features:
    print("-", feature)

In [ ]:
# Identify near-constant variables

near_constant = training_std.sort_values().head(30)

near_constant

In [ ]:
# Examine feature ranges

feature_ranges = pd.DataFrame({
    "min": all_training_values.min(),
    "max": all_training_values.max(),
    "mean": all_training_values.mean(),
    "std": all_training_values.std(),
})

feature_ranges["range"] = (
    feature_ranges["max"] - feature_ranges["min"]
)

feature_ranges.sort_values("range", ascending=False).head(20)

In [ ]:
# Distribution of sensor standard deviations

plt.figure(figsize=(10, 5))

plt.hist(
    training_std,
    bins=30,
)

plt.xlabel("Feature standard deviation")
plt.ylabel("Number of features")
plt.title("Distribution of Feature Variability")

plt.tight_layout()
plt.show()

In [ ]:
# Compute feature correlations

correlation_matrix = all_training_values.corr()

print("Correlation matrix shape:", correlation_matrix.shape)

correlation_matrix.head()

In [ ]:
# Visualize the correlation matrix

plt.figure(figsize=(14, 12))

plt.imshow(
    correlation_matrix,
    aspect="auto",
    interpolation="nearest",
)

plt.colorbar(label="Correlation")

plt.title("Sensor Correlation Matrix")
plt.xlabel("Sensor")
plt.ylabel("Sensor")

plt.tight_layout()
plt.show()

In [ ]:
# Find strongly correlated sensor pairs

corr = correlation_matrix.abs()

upper_triangle = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

strong_pairs = (
    upper_triangle
    .stack()
    .sort_values(ascending=False)
)

strong_pairs.head(30)

In [ ]:
# Compare feature means across dataset segments

segment_means = pd.DataFrame({
    "train1": training_segments[0][sensor_columns].mean(),
    "train2": training_segments[1][sensor_columns].mean(),
    "train3": training_segments[2][sensor_columns].mean(),
    "validation": validation_segments[0][sensor_columns].mean(),
})

segment_means.head(20)

In [ ]:
# Visualize the distribution of feature means across segments

plt.figure(figsize=(12, 6))

segment_means.boxplot()

plt.ylabel("Feature mean")
plt.title("Feature Mean Distribution Across Dataset Segments")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 16 — Attack statistics

for i, labels in enumerate(test_labels, 1):
    counts = labels.value_counts().sort_index()

    normal_count = counts.get(0, 0)
    attack_count = counts.get(1, 0)
    total_count = len(labels)

    print(f"\nTest {i}")
    print("-" * 50)
    print("Total timesteps:", total_count)
    print("Normal timesteps:", normal_count)
    print("Attack timesteps:", attack_count)
    print(
        "Attack percentage:",
        round(100 * attack_count / total_count, 2),
        "%"
    )

In [ ]:
# Cell 17 — Identify contiguous attack events

def find_attack_events(labels):
    labels = np.asarray(labels)

    attack_indices = np.where(labels == 1)[0]

    if len(attack_indices) == 0:
        return []

    events = []

    start = attack_indices[0]
    previous = attack_indices[0]

    for index in attack_indices[1:]:
        if index != previous + 1:
            events.append((start, previous))
            start = index

        previous = index

    events.append((start, previous))

    return events


for i, labels in enumerate(test_labels, 1):
    events = find_attack_events(labels)

    durations = [
        end - start + 1
        for start, end in events
    ]

    print(f"\nTest {i}")
    print("-" * 50)
    print("Number of attack events:", len(events))
    print("Total attack timesteps:", sum(durations))

    if durations:
        print("Minimum duration:", min(durations), "seconds")
        print("Maximum duration:", max(durations), "seconds")
        print("Median duration:", np.median(durations), "seconds")

In [ ]:
# Cell 18 — Visualize attack timelines

for i, (df, labels) in enumerate(
    zip(test_sensors, test_labels), 1
):
    plt.figure(figsize=(14, 3))

    plt.plot(
        df["timestamp"],
        labels,
    )

    plt.xlabel("Time")
    plt.ylabel("Attack label")
    plt.title(f"HAI Test {i} Attack Timeline")

    plt.yticks([0, 1], ["Normal", "Attack"])

    plt.tight_layout()
    plt.show()

## 2. Relationship Dynamics: Normal vs. Attack Behavior

This section investigates whether relationships between industrial
variables change during anomalous events.

The analysis is exploratory and is not used for model training,
hyperparameter tuning, or final test-set optimization.

In [5]:
# Cell 20 — Separate Test 1 normal and attack observations

test1_data = test_sensors[0].drop(columns="timestamp").copy()
test1_labels = test_labels[0].to_numpy()

normal_data = test1_data[test1_labels == 0]
attack_data = test1_data[test1_labels == 1]

print("Test 1 total observations:", len(test1_data))
print("Normal observations:", len(normal_data))
print("Attack observations:", len(attack_data))

print("\nNormal shape:", normal_data.shape)
print("Attack shape:", attack_data.shape)

Test 1 total observations: 54000
Normal observations: 51019
Attack observations: 2981

Normal shape: (51019, 86)
Attack shape: (2981, 86)


In [6]:
# Cell 21 — Correlation matrices for normal and attack periods

normal_correlation = normal_data.corr()
attack_correlation = attack_data.corr()

print("Normal correlation matrix:", normal_correlation.shape)
print("Attack correlation matrix:", attack_correlation.shape)

Normal correlation matrix: (86, 86)
Attack correlation matrix: (86, 86)


In [8]:
# Cell 22 — Diagnose undefined correlations

normal_constant = normal_data.columns[
    normal_data.nunique() <= 1
].tolist()

attack_constant = attack_data.columns[
    attack_data.nunique() <= 1
].tolist()

print("Constant features in normal data:", len(normal_constant))
print(normal_constant)

print("\nConstant features in attack data:", len(attack_constant))
print(attack_constant)

print("\nNaNs in normal correlation matrix:",
      normal_correlation.isna().sum().sum())

print("NaNs in attack correlation matrix:",
      attack_correlation.isna().sum().sum())

Constant features in normal data: 27
['P1_PIT01_HH', 'P1_PP01AD', 'P1_PP01AR', 'P1_PP01BD', 'P1_PP01BR', 'P1_PP02D', 'P1_PP02R', 'P1_PP04SP', 'P1_SOL01D', 'P1_SOL03D', 'P1_STSP', 'P2_ATSW_Lamp', 'P2_AutoGO', 'P2_Emerg', 'P2_MASW', 'P2_MASW_Lamp', 'P2_ManualGO', 'P2_ManualSD', 'P2_OnOff', 'P2_RTR', 'P2_TripEx', 'P2_VTR01', 'P2_VTR02', 'P2_VTR03', 'P2_VTR04', 'P3_LH01', 'P3_LL01']

Constant features in attack data: 27
['P1_PIT01_HH', 'P1_PP01AD', 'P1_PP01AR', 'P1_PP01BD', 'P1_PP01BR', 'P1_PP02D', 'P1_PP02R', 'P1_PP04SP', 'P1_SOL01D', 'P1_SOL03D', 'P1_STSP', 'P2_ATSW_Lamp', 'P2_AutoGO', 'P2_Emerg', 'P2_MASW', 'P2_MASW_Lamp', 'P2_ManualGO', 'P2_ManualSD', 'P2_OnOff', 'P2_RTR', 'P2_TripEx', 'P2_VTR01', 'P2_VTR02', 'P2_VTR03', 'P2_VTR04', 'P3_LH01', 'P3_LL01']

NaNs in normal correlation matrix: 3915
NaNs in attack correlation matrix: 3915


In [10]:
# Cell 23 — Compare sensor relationships using non-constant features

variable_features = [
    feature
    for feature in normal_data.columns
    if feature not in normal_constant
]

normal_correlation_variable = normal_data[
    variable_features
].corr()

attack_correlation_variable = attack_data[
    variable_features
].corr()

correlation_change = (
    attack_correlation_variable
    - normal_correlation_variable
).abs()

print("Variable features:", len(variable_features))
print(
    "Average absolute correlation change:",
    correlation_change.values.mean()
)
print(
    "Maximum absolute correlation change:",
    correlation_change.values.max()
)

Variable features: 59
Average absolute correlation change: 0.13603237774967464
Maximum absolute correlation change: 0.9798411796899258


In [11]:
# Cell 24 — Identify the sensor pairs with the largest correlation changes

change = correlation_change.copy()

upper_triangle = change.where(
    np.triu(
        np.ones(change.shape),
        k=1
    ).astype(bool)
)

largest_changes = (
    upper_triangle
    .stack()
    .sort_values(ascending=False)
)

print("Top 20 sensor pairs with the largest correlation changes:\n")

print(largest_changes.head(20))

Top 20 sensor pairs with the largest correlation changes:

P1_FCV03Z   P1_LIT01                 0.979841
P1_FCV03D   P1_LIT01                 0.968077
P1_FCV03Z   x1002_07_SETPOINT_OUT    0.920491
P1_FCV03D   x1002_07_SETPOINT_OUT    0.911187
P1_FCV03Z   P4_ST_PS                 0.898163
P1_FCV03D   P4_ST_PS                 0.885050
P1_FCV03Z   P4_HT_PS                 0.724454
P1_FCV03D   P4_HT_PS                 0.722789
P1_FT03     x1003_18_SETPOINT_OUT    0.696923
            P4_HT_PS                 0.690786
P1_FT03Z    P4_HT_PS                 0.669552
            x1003_18_SETPOINT_OUT    0.660724
P4_ST_TT01  x1002_08_SETPOINT_OUT    0.659171
P1_TIT01    x1002_08_SETPOINT_OUT    0.657525
P1_FT03     P4_ST_PS                 0.621240
P1_FT03Z    P4_ST_PS                 0.604696
P1_LCV01D   P1_TIT01                 0.599775
P1_LCV01Z   P1_TIT01                 0.599545
P1_LCV01D   P4_ST_TT01               0.598021
P1_LCV01Z   P4_ST_TT01               0.597626
dtype: float64


In [12]:
# Cell 25 — Inspect the strongest relationship changes

top_pairs = largest_changes.head(20)

comparison = pd.DataFrame({
    "normal_correlation": [
        normal_correlation_variable.loc[pair[0], pair[1]]
        for pair in top_pairs.index
    ],
    "attack_correlation": [
        attack_correlation_variable.loc[pair[0], pair[1]]
        for pair in top_pairs.index
    ],
    "absolute_change": top_pairs.values,
})

comparison.index = [
    f"{pair[0]} ↔ {pair[1]}"
    for pair in top_pairs.index
]

comparison

,normal_correlation,attack_correlation,absolute_change
P1_FCV03Z ↔ P1_LIT01,-0.692404,0.287438,0.979841
P1_FCV03D ↔ P1_LIT01,-0.696487,0.271590,0.968077
P1_FCV03Z ↔ x1002_07_SETPOINT_OUT,-0.597430,0.323061,0.920491
P1_FCV03D ↔ x1002_07_SETPOINT_OUT,-0.598104,0.313083,0.911187
P1_FCV03Z ↔ P4_ST_PS,0.346477,-0.551685,0.898163
P1_FCV03D ↔ P4_ST_PS,0.344492,-0.540558,0.885050
P1_FCV03Z ↔ P4_HT_PS,-0.078614,-0.803068,0.724454
P1_FCV03D ↔ P4_HT_PS,-0.071673,-0.794462,0.722789
P1_FT03 ↔ x1003_18_SETPOINT_OUT,0.621668,-0.075254,0.696923
P1_FT03 ↔ P4_HT_PS,-0.215307,-0.906093,0.690786


In [13]:
# Cell 26 — Inspect HAI's supplied process graph

from pathlib import Path
import json

GRAPH_ROOT = PROJECT_ROOT / "data" / "raw" / "HAI_Kaggle" / "graph" / "boiler"

graph_files = sorted(GRAPH_ROOT.glob("*.json"))

print("Graph files found:", len(graph_files))

for path in graph_files:
    print("-", path.name)

Graph files found: 8
- dcs_1001h.json
- dcs_1002h.json
- dcs_1003h.json
- dcs_1004h.json
- dcs_1010h.json
- dcs_1011h.json
- dcs_1020h.json
- phy_boiler.json


In [15]:
# Cell 27 — Inspect one HAI process graph

import ast

graph_path = GRAPH_ROOT / "dcs_1001h.json"

with open(graph_path, "r", encoding="utf-8") as f:
    graph_data = ast.literal_eval(f.read())

print("Top-level keys:")
print(graph_data.keys())

print("\nNumber of nodes:", len(graph_data["nodes"]))
print("Number of links:", len(graph_data["links"]))

print("\nFirst 10 nodes:")
for node in graph_data["nodes"][:10]:
    print(node)

print("\nFirst 10 links:")
for link in graph_data["links"][:10]:
    print(link)

Top-level keys:
dict_keys(['directed', 'multigraph', 'graph', 'nodes', 'links'])

Number of nodes: 31
Number of links: 42

First 10 nodes:
{'color': 'black', 'id': '1001.1'}
{'color': 'black', 'id': '1001.2'}
{'color': 'black', 'id': '1001.3'}
{'color': 'black', 'id': '1001.4'}
{'color': 'black', 'id': '1001.5'}
{'color': 'black', 'id': '1001.6'}
{'color': 'black', 'id': '1001.7'}
{'color': 'black', 'id': '1001.8'}
{'color': 'black', 'id': '1001.9'}
{'color': 'black', 'id': '1001.10'}

First 10 links:
{'label': 'DM-PIT01', 'source': '1001.1', 'target': '1001.2', 'key': 0}
{'label': 'DM-PIT01', 'source': '1001.1', 'target': '1001.3', 'key': 0}
{'label': 'DM-PIT01', 'source': '1001.1', 'target': '1001.16', 'key': 0}
{'label': '1001.2-OUT', 'source': '1001.2', 'target': '1001.11', 'key': 0}
{'label': 'TEST-AOUT', 'source': '1001.3', 'target': '1001.4', 'key': 0}
{'label': 'PV', 'source': '1001.4', 'target': 'HMI', 'key': 0}
{'label': '1001.5-OUT', 'source': '1001.5', 'target': '1001.14', 

In [16]:
# Cell 28 — Determine variable features using training data only

training_feature_names = [
    column
    for column in training_segments[0].columns
    if column != "timestamp"
]

training_constant_features = []

for feature in training_feature_names:
    is_constant = all(
        df[feature].nunique(dropna=False) <= 1
        for df in training_segments
    )

    if is_constant:
        training_constant_features.append(feature)

training_variable_features = [
    feature
    for feature in training_feature_names
    if feature not in training_constant_features
]

print("Total features:", len(training_feature_names))
print("Constant across all training segments:",
      len(training_constant_features))
print("Variable training features:",
      len(training_variable_features))

print("\nConstant training features:")
print(training_constant_features)

Total features: 86
Constant across all training segments: 20
Variable training features: 66

Constant training features:
['P1_PIT01_HH', 'P1_PP01AD', 'P1_PP01AR', 'P1_PP01BD', 'P1_PP01BR', 'P1_PP02D', 'P1_PP02R', 'P1_SOL01D', 'P1_SOL03D', 'P1_STSP', 'P2_Emerg', 'P2_OnOff', 'P2_RTR', 'P2_TripEx', 'P2_VTR01', 'P2_VTR02', 'P2_VTR03', 'P2_VTR04', 'P3_LH01', 'P3_LL01']


In [18]:
# Cell 29 — Training-only correlation matrix for graph construction

training_variable_data = pd.concat(
    [
        df[training_variable_features]
        for df in training_segments
    ],
    axis=0,
    ignore_index=True,
)

training_correlation = training_variable_data.corr()

print(
    "Training correlation matrix:",
    training_correlation.shape
)

print(
    "NaNs in training correlation matrix:",
    training_correlation.isna().sum().sum()
)

print(
    "Minimum correlation:",
    training_correlation.min().min()
)

print(
    "Maximum correlation:",
    training_correlation.max().max()
)

Training correlation matrix: (66, 66)
NaNs in training correlation matrix: 0
Minimum correlation: -0.9999999999999987
Maximum correlation: 1.0


In [19]:
# Cell 30 — Analyze the distribution of training correlations

upper_triangle = training_correlation.where(
    np.triu(
        np.ones(training_correlation.shape),
        k=1
    ).astype(bool)
)

absolute_correlations = (
    upper_triangle
    .abs()
    .stack()
)

print("Number of unique feature pairs:", len(absolute_correlations))

print("\nAbsolute correlation quantiles:")
print(
    absolute_correlations.quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nNumber of pairs above thresholds:")

for threshold in [0.3, 0.5, 0.7, 0.8, 0.9, 0.95]:
    count = (absolute_correlations >= threshold).sum()
    print(
        f"|r| >= {threshold:.2f}: "
        f"{count} pairs"
    )

Number of unique feature pairs: 4356

Absolute correlation quantiles:
0.50    0.029682
0.75    0.103779
0.90    0.358705
0.95    0.639921
0.99    0.980905
dtype: float64

Number of pairs above thresholds:
|r| >= 0.30: 261 pairs
|r| >= 0.50: 145 pairs
|r| >= 0.70: 100 pairs
|r| >= 0.80: 67 pairs
|r| >= 0.90: 50 pairs
|r| >= 0.95: 40 pairs


In [21]:
# Cell 31 — Inspect strongest training relationships for each feature

absolute_training_correlation = training_correlation.abs().copy()

# Make a writable NumPy copy and remove self-correlations
correlation_values = absolute_training_correlation.to_numpy(copy=True)
np.fill_diagonal(correlation_values, np.nan)

absolute_training_correlation = pd.DataFrame(
    correlation_values,
    index=training_correlation.index,
    columns=training_correlation.columns,
)

top_k = 5

feature_connectivity = {}

for feature in training_variable_features:
    strongest = (
        absolute_training_correlation[feature]
        .dropna()
        .sort_values(ascending=False)
        .head(top_k)
    )

    feature_connectivity[feature] = strongest

for feature in training_variable_features[:10]:
    print(f"\n{feature}")
    print(feature_connectivity[feature])


P1_FCV01D
P1_FCV01Z                0.998587
P1_FT02                  0.980412
x1003_10_SETPOINT_OUT    0.963573
P1_FT02Z                 0.963496
P1_TIT03                 0.793420
Name: P1_FCV01D, dtype: float64

P1_FCV01Z
P1_FCV01D                0.998587
P1_FT02                  0.981539
P1_FT02Z                 0.965674
x1003_10_SETPOINT_OUT    0.965140
P1_TIT03                 0.794553
Name: P1_FCV01Z, dtype: float64

P1_FCV02D
P1_FCV02Z                0.905364
x1003_10_SETPOINT_OUT    0.827345
P1_FT02Z                 0.827223
P1_TIT02                 0.798292
P1_FCV01D                0.752329
Name: P1_FCV02D, dtype: float64

P1_FCV02Z
P1_FCV02D                0.905364
P1_FT02Z                 0.871534
x1003_10_SETPOINT_OUT    0.871392
P1_TIT02                 0.855314
P1_FCV01Z                0.772048
Name: P1_FCV02Z, dtype: float64

P1_FCV03D
P1_FCV03Z                0.989090
x1002_08_SETPOINT_OUT    0.734346
P1_FT03                  0.725452
P1_FT03Z                 0.722227
P

In [22]:
# Cell 32 — Evaluate candidate correlation thresholds

thresholds = [0.3, 0.5, 0.7, 0.8, 0.9, 0.95]

for threshold in thresholds:
    adjacency = (
        absolute_training_correlation >= threshold
    ).astype(int)

    degree = adjacency.sum(axis=1)

    print(f"\n|r| >= {threshold:.2f}")
    print("  Minimum degree:", degree.min())
    print("  Maximum degree:", degree.max())
    print("  Mean degree:", round(degree.mean(), 2))
    print("  Isolated nodes:", (degree == 0).sum())


|r| >= 0.30
  Minimum degree: 0
  Maximum degree: 18
  Mean degree: 7.91
  Isolated nodes: 5

|r| >= 0.50
  Minimum degree: 0
  Maximum degree: 11
  Mean degree: 4.39
  Isolated nodes: 14

|r| >= 0.70
  Minimum degree: 0
  Maximum degree: 8
  Mean degree: 3.03
  Isolated nodes: 15

|r| >= 0.80
  Minimum degree: 0
  Maximum degree: 7
  Mean degree: 2.03
  Isolated nodes: 20

|r| >= 0.90
  Minimum degree: 0
  Maximum degree: 6
  Mean degree: 1.52
  Isolated nodes: 25

|r| >= 0.95
  Minimum degree: 0
  Maximum degree: 6
  Mean degree: 1.21
  Isolated nodes: 32


In [23]:
# Cell 33 — Evaluate candidate top-k graph densities

candidate_k = [3, 5, 8, 10]

for k in candidate_k:
    adjacency = pd.DataFrame(
        0,
        index=training_variable_features,
        columns=training_variable_features,
        dtype=int,
    )

    for feature in training_variable_features:
        neighbors = (
            absolute_training_correlation[feature]
            .dropna()
            .nlargest(k)
            .index
        )

        adjacency.loc[feature, neighbors] = 1

    # Make the graph undirected:
    adjacency = ((adjacency + adjacency.T) > 0).astype(int)

    degree = adjacency.sum(axis=1)

    print(f"\nk = {k}")
    print("  Minimum degree:", degree.min())
    print("  Maximum degree:", degree.max())
    print("  Mean degree:", round(degree.mean(), 2))
    print("  Isolated nodes:", (degree == 0).sum())
    print("  Total undirected edges:", int(degree.sum() / 2))


k = 3
  Minimum degree: 3
  Maximum degree: 8
  Mean degree: 4.36
  Isolated nodes: 0
  Total undirected edges: 144

k = 5
  Minimum degree: 5
  Maximum degree: 11
  Mean degree: 6.73
  Isolated nodes: 0
  Total undirected edges: 222

k = 8
  Minimum degree: 8
  Maximum degree: 18
  Mean degree: 10.82
  Isolated nodes: 0
  Total undirected edges: 357

k = 10
  Minimum degree: 10
  Maximum degree: 23
  Mean degree: 13.09
  Isolated nodes: 0
  Total undirected edges: 432
